# Tutorial for Downloading PACE Dataset
PACE (Pose Annotations in Cluttered Environments) is a large-scale benchmark for 6D pose estimation and tracking. This notebook downloads the dataset from Hugging Face and prepares it for evaluation following the dataset's guidance.

- Project repo: [PACE GitHub](https://github.com/qq456cvb/PACE#dataset-download)
- Dataset host: [PACE on Hugging Face](https://huggingface.co/datasets/qq456cvb/PACE/tree/main)


In [ ]:
%pip install --quiet requests tqdm huggingface_hub

### 1. Set up the download configuration
Set the base save path and (optional) proxy. Large files on Hugging Face may be split; we'll automatically merge chunks like `*_chunk_*` into a single `tar.gz` before extracting.


In [ ]:
############################## Set the configuration for loading data ###########################
DATA_PATH = 'data/PACE/'                # Path to the directory where the data will be stored
# PROXY = 'http://proxy-server:proxy-port'   # Proxy server to use for downloading the data (or None)
PROXY = None
SELECTED_SPLITS = [
    # Choose which archives to download; leave empty to download everything found.
    # Examples: 'models.tar.gz', 'models_eval.tar.gz', 'test.tar.gz', 'val_inst.tar.gz', 'train_pbr_inst.tar.gz'
]
# If you know you only want real test set for evaluation:
# SELECTED_SPLITS = ['test.tar.gz']

# Behavior toggles
MERGE_CHUNKS = True          # Merge files matching *chunk_* into a single tar.gz
EXTRACT_ARCHIVES = True      # Extract downloaded tar.gz archives
REMOVE_CHUNKS = True         # Remove chunk files after merging
REMOVE_TARS = False          # Remove tar.gz files after extraction
#################################################################################################

import os, sys, json, re, shutil
from pathlib import Path
os.makedirs(DATA_PATH, exist_ok=True)

# Set the proxy server mapping (compatible with requests)
proxies = {'http': PROXY, 'https': PROXY} if PROXY else None



### 2. Discover available files on Hugging Face
We list files from the dataset repository and select relevant archives. Large items may appear as `*_chunk_*` parts. We'll normalize names so that chunks grouped under their final archive name.


In [2]:
from huggingface_hub import list_repo_files
import requests
from tqdm import tqdm

HF_REPO_ID = 'qq456cvb/PACE'

# File patterns of interest
ARCHIVE_SUFFIXES = ('.tar.gz',)
CHUNK_PATTERN = re.compile(r'(.+?)_chunk_\w+$')


def list_pace_files(repo_id: str = HF_REPO_ID):
    files = list_repo_files(repo_id, repo_type='dataset')
    # Keep only tar.gz files and their chunks
    candidates = [f for f in files if f.endswith('.tar.gz') or '_chunk_' in f]
    return sorted(candidates)


def group_chunks(files):
    groups = {}
    for f in files:
        base = f
        m = CHUNK_PATTERN.match(Path(f).name)
        if m:
            base = m.group(1) + '.tar.gz'
        groups.setdefault(base, []).append(f)
    # Ensure deterministic order per group
    for k in groups:
        groups[k] = sorted(groups[k])
    return groups


def select_archives(groups, selected_names):
    if not selected_names:
        return groups
    selected = {}
    for name in selected_names:
        if name in groups:
            selected[name] = groups[name]
        else:
            # If a direct name not found, try exact file listing
            matching = {k: v for k, v in groups.items() if Path(k).name == name}
            selected.update(matching)
    return selected

files = list_pace_files()
file_groups = group_chunks(files)
selected_groups = select_archives(file_groups, SELECTED_SPLITS)

print(f"Found {len(files)} candidate files on Hugging Face.")
print(f"Will download/extract {len(selected_groups)} archives:")
for k, v in selected_groups.items():
    print(f" - {k} {'(chunked)' if any('_chunk_' in x for x in v) else ''}")


Found 25 candidate files on Hugging Face.
Will download/extract 8 archives:
 - models.tar.gz 
 - models_eval.tar.gz 
 - models_nocs.tar.gz 
 - test.tar.gz (chunked)
 - train_pbr_cat.tar.gz (chunked)
 - train_pbr_inst.tar.gz (chunked)
 - val_inst.tar.gz 
 - val_pbr_cat.tar.gz 


### 3. Download selected archives
We will download all selected archives (or all discovered) from Hugging Face. If a file already exists and is complete, it will be skipped. Partial files will resume via HTTP Range requests.


In [ ]:
from urllib.parse import quote

BASE_DOWNLOAD_URL = f"https://huggingface.co/datasets/{HF_REPO_ID}/resolve/main"


def hf_file_url(relpath: str) -> str:
    relpath = relpath.lstrip('/')
    return f"{BASE_DOWNLOAD_URL}/{quote(relpath)}?download=true"


def download_with_retries(url, proxies=None, headers=None, timeout=10, max_retries=5, retry_delay=2):
    if headers is None:
        headers = {}
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, stream=True, proxies=proxies, timeout=timeout)
            response.raise_for_status()
            return response
        except (requests.Timeout, requests.ConnectionError):
            print(f"Request timed out or connection error, attempting retry {attempt + 1}...")
            from time import sleep
            sleep(retry_delay)
        except requests.HTTPError as err:
            print(f"HTTP error: {err}")
            return None
    print("All retry attempts failed")
    return None


def download_file(url: str, filename: str) -> bool:
    progress_bar = None
    try:
        parent = os.path.dirname(filename)
        os.makedirs(parent, exist_ok=True)
        hidden_marker = os.path.join(parent, '.' + os.path.basename(filename) + '.done')
        if os.path.exists(hidden_marker):
            print(f"{filename} already exists!")
            return True
        elif os.path.exists(filename):
            print('\033[93m' + f"{filename} already exists but is incomplete! Resuming download..." + '\033[0m')
            resume_byte_pos = os.path.getsize(filename)
        else:
            resume_byte_pos = 0
            print('\033[93m' + f"Downloading {filename}..." + '\033[0m')

        headers = {'Range': f'bytes={resume_byte_pos}-'} if resume_byte_pos else {}
        response = download_with_retries(url, headers=headers, proxies=proxies)
        if response is None:
            return False
        total_size_in_bytes = int(response.headers.get('content-length', 0))
        progress_bar = tqdm(total=total_size_in_bytes, unit='iB', unit_scale=True)
        if response.status_code in (200, 206):
            mode = 'ab' if response.status_code == 206 else 'wb'
            with open(filename, mode) as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        progress_bar.update(len(chunk))
                        f.write(chunk)
            progress_bar.close()
            with open(hidden_marker, 'w') as f:
                f.write('done')
            return True
        else:
            print('\033[91m' + f"Failed to download {filename}! The server returned status code {response.status_code}" + '\033[0m')
            return False
    except (KeyboardInterrupt, requests.RequestException, Exception) as e:
        print('\033[91m' + f"An error occurred while downloading {filename}! {e}" + '\033[0m')
        return False
    finally:
        if progress_bar is not None:
            progress_bar.close()


fail_list = []
for archive_name, paths in selected_groups.items():
    print('----------------------------------------------------------------------------------------------------')
    print('\033[93m' + f"Preparing to download archive group: {archive_name}" + '\033[0m')
    for relpath in paths:
        url = hf_file_url(relpath)
        out_path = os.path.join(DATA_PATH, os.path.basename(relpath))
        ok = download_file(url, out_path)
        if not ok:
            fail_list.append(relpath)

if len(fail_list) == 0:
    print('\033[92m' + "All selected files downloaded successfully!" + '\033[0m')
else:
    print('\n' + '-'*100)
    print('\033[91m' + "Download failed for the following files:" + '\033[0m')
    print(fail_list)
    print('\033[91m' + "Please rerun the cell to retry downloading the failed files." + '\033[0m')


### 4. Merge split chunk files (if any)
If an archive is split into `*_chunk_*`, we concatenate them in order into the final `*.tar.gz` file.


In [8]:

def merge_chunks_for_group(archive_name: str, relpaths: list):
    chunk_basenames = [os.path.basename(p) for p in relpaths if '_chunk_' in p]
    if not chunk_basenames:
        return None
    dest_tar = os.path.join(DATA_PATH, os.path.basename(archive_name))
    marker = os.path.join(DATA_PATH, '.' + os.path.basename(dest_tar) + '.merged')
    if os.path.exists(marker):
        print(f"{dest_tar} already merged!")
        return dest_tar

    # Ensure all chunks are present locally
    local_chunks = [os.path.join(DATA_PATH, bn) for bn in sorted(chunk_basenames)]
    missing = [p for p in local_chunks if not os.path.exists(p)]
    if missing:
        print('\033[91m' + f"Missing chunk files; cannot merge: {missing}" + '\033[0m')
        return None

    print('\033[93m' + f"Merging {len(local_chunks)} chunks into {dest_tar} ..." + '\033[0m')
    with open(dest_tar, 'wb') as out_f:
        for chunk_path in local_chunks:
            with open(chunk_path, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)
    with open(marker, 'w') as f:
        f.write('merged')

    if REMOVE_CHUNKS:
        for chunk_path in local_chunks:
            try:
                os.remove(chunk_path)
            except Exception:
                pass
    return dest_tar


merged_map = {}
if MERGE_CHUNKS:
    for archive_name, relpaths in selected_groups.items():
        merged = merge_chunks_for_group(archive_name, relpaths)
        if merged:
            merged_map[archive_name] = merged
else:
    print('Skipping chunk merging as MERGE_CHUNKS=False')

print('Merge step complete.')


Merging 2 chunks into dataset/PACE/test.tar.gz ...
Merge step complete.


### 5. Extract archives into `data/PACE/`
We will extract the resulting `*.tar.gz` into the target directory.


In [9]:
import tarfile


def is_within_directory(directory: str, target: str) -> bool:
    directory = os.path.realpath(directory)
    target = os.path.realpath(target)
    return os.path.commonprefix([directory, target]) == directory


def safe_extract_tar(tar: tarfile.TarFile, path: str = "."):
    for member in tar.getmembers():
        member_path = os.path.join(path, member.name)
        if not is_within_directory(path, member_path):
            raise Exception("Attempted Path Traversal in Tar File")
    tar.extractall(path)


def local_archive_path_for_group(archive_name: str, relpaths: list):
    # Prefer merged tar if exists; else the single tar
    if MERGE_CHUNKS and archive_name in merged_map:
        return merged_map[archive_name]
    # If not chunked, expect a single file named exactly the archive
    basename = os.path.basename(archive_name)
    path = os.path.join(DATA_PATH, basename)
    if os.path.exists(path):
        return path
    # Fallback: if the group contains exactly one non-chunk file, use it
    non_chunks = [p for p in relpaths if '_chunk_' not in p]
    if len(non_chunks) == 1:
        alt = os.path.join(DATA_PATH, os.path.basename(non_chunks[0]))
        if os.path.exists(alt):
            return alt
    return None


fail_extract = []
if EXTRACT_ARCHIVES:
    for archive_name, relpaths in selected_groups.items():
        local_tar = local_archive_path_for_group(archive_name, relpaths)
        if not local_tar or not local_tar.endswith('.tar.gz') or not os.path.exists(local_tar):
            print('\033[93m' + f"Skipping extraction for {archive_name}; no local tar found." + '\033[0m')
            continue
        print('\033[93m' + f"Extracting {os.path.basename(local_tar)} to {DATA_PATH} ..." + '\033[0m')
        try:
            with tarfile.open(local_tar, 'r:gz') as tar:
                safe_extract_tar(tar, DATA_PATH)
            if REMOVE_TARS:
                try:
                    os.remove(local_tar)
                except Exception:
                    pass
        except Exception as e:
            print('\033[91m' + f"Extraction failed for {local_tar}: {e}" + '\033[0m')
            fail_extract.append(local_tar)
    if not fail_extract:
        print('\033[92m' + "All archives extracted successfully!" + '\033[0m')
    else:
        print('\n' + '-'*100)
        print('\033[91m' + "Extraction failed for the following archives:" + '\033[0m')
        print(fail_extract)
else:
    print('Skipping extraction as EXTRACT_ARCHIVES=False')


Extracting models_eval.tar.gz to dataset/PACE/ ...
Extracting test.tar.gz to dataset/PACE/ ...
All archives extracted successfully!
